# QuantStrata Machine Learning Pipeline — TensorFlow Tutorial

This comprehensive tutorial demonstrates the QuantStrata ML framework, a production-grade TensorFlow-native pipeline for quantitative finance applications.

## What You'll Learn

| Section | Topics |
|---------|--------|
| **1. Data Preparation** | TFDataset, normalization, train/val/test splits, tf.data.Dataset |
| **2. Model Architecture** | MLPPricer, PricingModel base class, custom architectures |
| **3. Training** | Trainer class, TrainingConfig, callbacks, early stopping |
| **4. Evaluation** | Evaluator, metrics (MSE, MAE, R², MAPE), visualization |
| **5. Inference** | SavedModel export, loading, Predictor class |
| **6. Advanced Topics** | Greeks via autodiff, uncertainty estimation, ensembles |

---

## Prerequisites

Install dependencies from the project root:

```bash
pip install -r requirements.txt
# or at least: pip install tensorflow numpy pandas matplotlib scipy
```

The tutorial uses **TensorFlow 2.x** and the QuantStrata ML module.

**Important:** Run the **first code cell (Setup and imports)** before running any other cell. It adds the project root to `sys.path` so that `src.m_learning` imports work.

## Overview: What We're Building

**Goal:** Train a neural network to approximate the **option price** as a function of contract and market parameters. Once trained, the model gives fast prices without running Monte Carlo or closed-form formulae repeatedly.

**Inputs (6 features):**

| Feature | Symbol | Description |
|--------|--------|-------------|
| Spot | \(S\) | Underlying price |
| Strike | \(K\) | Strike price |
| Volatility | \(\sigma\) | Volatility (e.g. implied vol) |
| Rate | \(r\) | Risk-free rate |
| Time to expiry | \(T\) | Time to maturity (years) |
| Option type | call/put | +1 for call, 0 for put |

**Target:** A single scalar — the **option price** \(P\).

**Mathematical view:** In theory the price satisfies a pricing equation (e.g. Black–Scholes \(P = S \Phi(d_1) - K e^{-rT} \Phi(d_2)\) for a call). We do not implement that formula in the network; instead we **learn** the mapping \((S, K, \sigma, r, T, \text{type}) \mapsto P\) from data (e.g. prices from Black–Scholes or Monte Carlo). A multilayer perceptron (MLP) can approximate this smooth function arbitrarily well (universal approximation). We use normalized inputs and optionally normalized targets for stable training.

In [ ]:
# Setup and imports — run this cell first before any other cells
import sys
from pathlib import Path

# Find project root (directory containing src/m_learning)
def _find_project_root():
    path = Path.cwd()
    for _ in range(6):
        if (path / "src" / "m_learning").exists():
            return path
        if path.parent == path:
            break
        path = path.parent
    # Fallback: assume notebook is in docs/tutorials/m_learning
    return Path.cwd().parents[2]

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

# Set style for better plots (fallback if seaborn style missing)
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"Project root: {project_root}")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

---

# 1. Data Preparation

The ML pipeline starts with data preparation. The `TFDataset` class provides a unified interface for:

- **Feature/target storage** with NumPy arrays
- **Normalization** (z-score or min-max scaling)
- **Train/val/test splitting**
- **Conversion to `tf.data.Dataset`** for efficient training

## 1.1 Generating Synthetic Data

For this tutorial, we'll generate synthetic option pricing data. The `create_pricing_dataset` function:

1. Samples random option parameters (spot, strike, vol, rate, expiry, type)
2. Computes Black-Scholes prices as targets
3. Returns a `TFDataset` with features and targets

In [ ]:
from src.m_learning.data import create_pricing_dataset, TFDataset

# Generate 20,000 samples
dataset = create_pricing_dataset(
    n_samples=20000,
    spot_range=(80, 120),      # Underlying price range
    strike_range=(80, 120),    # Strike price range
    vol_range=(0.10, 0.50),    # Volatility range (10% - 50%)
    rate_range=(0.01, 0.10),   # Risk-free rate (1% - 10%)
    expiry_range=(0.1, 2.0),   # Time to expiry (0.1 - 2 years)
    seed=42,
)

print(f"Dataset: {dataset}")
print(f"\nFeature names: {dataset.feature_names}")
print(f"Target names: {dataset.target_names}")
print(f"\nMetadata: {dataset.metadata}")

In [ ]:
# Examine the data
features_df = pd.DataFrame(dataset.features, columns=dataset.feature_names)
features_df['price'] = dataset.targets.flatten()

print("Data Statistics:")
print(features_df.describe().round(3))

## 1.2 Visualizing the Data

Let's explore the distribution of features and their relationship to option prices.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold')

feature_names = ['spot', 'strike', 'volatility', 'rate', 'time_to_expiry', 'is_call']
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B', '#95C623']

for i, (ax, name, color) in enumerate(zip(axes.flat, feature_names, colors)):
    if name == 'is_call':
        # Bar chart for categorical
        counts = features_df[name].value_counts().sort_index()
        ax.bar(['Put (0)', 'Call (1)'], counts.values, color=color, alpha=0.7)
        ax.set_ylabel('Count')
    else:
        # Histogram for continuous
        ax.hist(features_df[name], bins=50, color=color, alpha=0.7, edgecolor='white')
        ax.set_ylabel('Frequency')
    
    ax.set_xlabel(name.replace('_', ' ').title())
    ax.set_title(f'{name}', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Price distribution and relationship to key features
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Price distribution
ax = axes[0]
ax.hist(features_df['price'], bins=50, color='#2E86AB', alpha=0.7, edgecolor='white')
ax.axvline(features_df['price'].mean(), color='red', linestyle='--', label=f'Mean: ${features_df["price"].mean():.2f}')
ax.axvline(features_df['price'].median(), color='green', linestyle='--', label=f'Median: ${features_df["price"].median():.2f}')
ax.set_xlabel('Option Price ($)')
ax.set_ylabel('Frequency')
ax.set_title('Price Distribution')
ax.legend()

# 2. Price vs Moneyness
ax = axes[1]
moneyness = features_df['spot'] / features_df['strike']
calls = features_df['is_call'] > 0.5
ax.scatter(moneyness[calls], features_df['price'][calls], alpha=0.3, s=5, label='Call', color='#2E86AB')
ax.scatter(moneyness[~calls], features_df['price'][~calls], alpha=0.3, s=5, label='Put', color='#A23B72')
ax.set_xlabel('Moneyness (S/K)')
ax.set_ylabel('Option Price ($)')
ax.set_title('Price vs Moneyness')
ax.legend()

# 3. Price vs Volatility
ax = axes[2]
scatter = ax.scatter(features_df['volatility'], features_df['price'], 
                     c=features_df['time_to_expiry'], alpha=0.3, s=5, cmap='viridis')
plt.colorbar(scatter, ax=ax, label='Time to Expiry')
ax.set_xlabel('Volatility')
ax.set_ylabel('Option Price ($)')
ax.set_title('Price vs Volatility (colored by expiry)')

plt.tight_layout()
plt.show()

## 1.3 Normalization

Neural networks train better with normalized data. The `TFDataset` class provides:

- **Z-score normalization**: `(x - mean) / std` → mean=0, std=1
- **Min-max normalization**: `(x - min) / (max - min)` → range [0, 1]

The normalization statistics are saved and can be used to denormalize predictions later.

In [ ]:
# Normalize features and targets
dataset.normalize_features(method='zscore')
dataset.normalize_targets(method='zscore')

print("Normalization Statistics:")
print(f"\nFeature means: {dataset.feature_stats.mean.round(3)}")
print(f"Feature stds:  {dataset.feature_stats.std.round(3)}")
print(f"\nTarget mean: {dataset.target_stats.mean[0]:.3f}")
print(f"Target std:  {dataset.target_stats.std[0]:.3f}")

# Verify normalization
print(f"\nAfter normalization:")
print(f"Feature mean (should be ~0): {dataset.features.mean(axis=0).round(3)}")
print(f"Feature std (should be ~1): {dataset.features.std(axis=0).round(3)}")

## 1.4 Train/Validation/Test Split

We split the data into:
- **Training set (70%)**: Used to update model weights
- **Validation set (15%)**: Used for early stopping and hyperparameter tuning
- **Test set (15%)**: Held out for final evaluation

In [ ]:
# Split the data
train_data, val_data, test_data = dataset.split(
    train=0.70,
    val=0.15,
    test=0.15,
    seed=42,
)

print(f"Training samples:   {len(train_data):,}")
print(f"Validation samples: {len(val_data):,}")
print(f"Test samples:       {len(test_data):,}")

## 1.5 Creating tf.data.Dataset

For efficient training, we convert to `tf.data.Dataset` with:
- **Batching**: Process multiple samples at once
- **Shuffling**: Randomize order each epoch
- **Prefetching**: Load next batch while GPU processes current

In [ ]:
BATCH_SIZE = 256

# Create tf.data.Dataset objects
train_ds = train_data.to_tf_dataset(batch_size=BATCH_SIZE, shuffle=True)
val_ds = val_data.to_tf_dataset(batch_size=BATCH_SIZE, shuffle=False)
test_ds = test_data.to_tf_dataset(batch_size=BATCH_SIZE, shuffle=False)

# Inspect one batch
for features, targets in train_ds.take(1):
    print(f"Batch shape: features={features.shape}, targets={targets.shape}")
    print(f"Feature dtype: {features.dtype}")
    print(f"Target dtype: {targets.dtype}")

## Plug-and-Play Design

The same **training → evaluation → inference** pipeline works for any model that conforms to the framework:

| Component | Interface | What you need |
|-----------|-----------|----------------|
| **Data** | `TFDataset` or `(X, y)` or `tf.data.Dataset` | Features and targets (NumPy or dataset) |
| **Model** | Any `tf.keras.Model` | Implement `call(inputs, training=...)`; optional: subclass `PricingModel` for metadata and helpers |
| **Training** | `Trainer(model, config).fit(train_data, val_data)` | Same for all models |
| **Evaluation** | `evaluate_model(model, data)` or `Evaluator(model).evaluate(data)` | Same metrics and plots |
| **Inference** | `save_model(model, path)`, `load_model(path)`, `Predictor(model)` | Same artifact layout; use `custom_objects` when loading custom classes |

So you can swap in a deeper MLP, a residual net, or another `tf.keras.Model` and still use `Trainer`, `Evaluator`, `save_model`/`load_model`, and `Predictor` without changing the pipeline. Subclassing `BaseModel`/`PricingModel` is optional and adds metadata + optional methods (e.g. `price_with_greeks`).

---

# 2. Model Architecture

The QuantStrata ML module provides a `PricingModel` base class that extends `tf.keras.Model` with:

- **Metadata tracking**: Automatically records model configuration
- **Greeks computation**: Automatic differentiation for sensitivities
- **Standard interface**: Consistent API across all pricing models

## 2.1 MLPPricer Architecture

The `MLPPricer` is a Multi-Layer Perceptron designed for option pricing:

```
Input (6 features)
    │
    ▼
┌─────────────────┐
│ Dense (128)     │ ← ReLU activation
│ BatchNorm       │ ← Training stability
│ Dropout (0.1)   │ ← Regularization
└─────────────────┘
    │
    ▼
┌─────────────────┐
│ Dense (64)      │
│ BatchNorm       │
│ Dropout (0.1)   │
└─────────────────┘
    │
    ▼
┌─────────────────┐
│ Dense (32)      │
│ BatchNorm       │
│ Dropout (0.1)   │
└─────────────────┘
    │
    ▼
Output (1 price)
```

**Mathematical view:** The MLP computes a composition of affine maps and nonlinearities: \(y_0 = x\), then \(y_{k+1} = \phi(W_k y_k + b_k)\) for hidden layers (e.g. \(\phi = \mathrm{ReLU}\)), and finally \(\hat{P} = W_{\mathrm{out}} y_L + b_{\mathrm{out}}\). We train by minimizing mean squared error \(\frac{1}{n}\sum_i (\hat{P}_i - P_i)^2\) over the training set. This learns an approximation to the true pricing function; with enough width/depth, an MLP can approximate the smooth Black–Scholes (or other) mapping arbitrarily well.

In [ ]:
from src.m_learning.models import MLPPricer, create_mlp_pricer

# Create model using factory function (recommended)
model = create_mlp_pricer(
    n_features=6,                    # spot, strike, vol, rate, expiry, type
    hidden_units=[128, 64, 32],      # 3 hidden layers
    activation='relu',               # ReLU activation
    dropout_rate=0.1,                # 10% dropout
    use_batch_norm=True,             # Batch normalization
    name='fx_vanilla_pricer',
)

# Model summary
model.summary()

In [ ]:
# Inspect model metadata
print("Model Metadata:")
for key, value in model.metadata.items():
    print(f"  {key}: {value}")

---

# 3. Training

The `Trainer` class provides a high-level interface for training with:

- **Configuration management**: All settings in `TrainingConfig`
- **Automatic callbacks**: Early stopping, checkpointing, logging
- **Progress tracking**: Training curves, best epoch detection

## 3.1 Training Configuration

In [ ]:
from src.m_learning.core import (
    TrainingConfig,
    OptimizerConfig,
    EarlyStoppingConfig,
)
from src.m_learning.training import Trainer

# Configure training
config = TrainingConfig(
    epochs=150,                       # Maximum epochs
    batch_size=BATCH_SIZE,
    
    # Optimizer: Adam with learning rate 0.001
    optimizer=OptimizerConfig(
        name='adam',
        learning_rate=1e-3,
    ),
    
    # Early stopping: stop if val_loss doesn't improve for 15 epochs
    early_stopping=EarlyStoppingConfig(
        patience=15,
        min_delta=1e-5,
        monitor='val_loss',
        restore_best_weights=True,
    ),
    
    # Loss and metrics
    loss='mse',
    metrics=['mae'],
    
    # Verbosity
    verbose=1,
    seed=42,
)

print("Training Configuration:")
for key, value in config.to_dict().items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for k, v in value.items():
            print(f"    {k}: {v}")
    else:
        print(f"  {key}: {value}")

## 3.2 Running Training

The `Trainer.fit()` method:
1. Compiles the model with the specified optimizer and loss
2. Configures callbacks (early stopping, progress logging)
3. Trains the model
4. Returns a `TrainingResult` with history and metrics

In [ ]:
# Create trainer
trainer = Trainer(model, config)

# Train the model
print("Starting training...\n")
result = trainer.fit(train_data, val_data)

print(f"\n{'='*50}")
print(f"Training Complete!")
print(f"{'='*50}")
print(f"Final epoch: {result.final_epoch}")
print(f"Best epoch: {result.best_epoch}")
print(f"Best train loss: {result.best_train_loss:.6f}")
print(f"Best val loss: {result.best_val_loss:.6f}")
print(f"Stopped early: {result.stopped_early}")
print(f"Training time: {result.total_time_seconds:.1f}s")

## 3.3 Training Curves

Visualizing training and validation loss helps diagnose:
- **Overfitting**: Train loss decreases but val loss increases
- **Underfitting**: Both losses plateau at high values
- **Good fit**: Both losses decrease and converge

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(result.history['loss']) + 1)

# Loss curves
ax = axes[0]
ax.plot(epochs, result.history['loss'], 'b-', linewidth=2, label='Training Loss')
ax.plot(epochs, result.history['val_loss'], 'r-', linewidth=2, label='Validation Loss')
ax.axvline(result.best_epoch, color='green', linestyle='--', alpha=0.7, 
           label=f'Best Epoch ({result.best_epoch})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MSE)')
ax.set_title('Training and Validation Loss', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')  # Log scale to see improvement

# MAE curves
ax = axes[1]
ax.plot(epochs, result.history['mae'], 'b-', linewidth=2, label='Training MAE')
ax.plot(epochs, result.history['val_mae'], 'r-', linewidth=2, label='Validation MAE')
ax.axvline(result.best_epoch, color='green', linestyle='--', alpha=0.7,
           label=f'Best Epoch ({result.best_epoch})')
ax.set_xlabel('Epoch')
ax.set_ylabel('MAE (Normalized)')
ax.set_title('Training and Validation MAE', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# 4. Evaluation

The `Evaluator` class provides comprehensive model evaluation with:

- **Standard metrics**: MSE, MAE, RMSE, R², MAPE
- **Domain-specific metrics**: Max error, percentile errors
- **Visualization**: Prediction plots, residual analysis, error distributions

## 4.1 Test Set Evaluation

In [ ]:
from src.m_learning.evaluation import Evaluator, evaluate_model

# Create evaluator with target scaler for denormalization
evaluator = Evaluator(model, target_scaler=dataset.target_stats)

# Evaluate on test set
eval_result = evaluator.evaluate(
    test_data,
    metrics=['mse', 'mae', 'rmse', 'mape', 'r2', 'max_error', 'p95_error'],
)

print(eval_result.summary())

## 4.2 Prediction Visualization

Visualizing predictions vs actual values reveals model performance patterns.

In [ ]:
# Comprehensive prediction visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

y_true = eval_result.targets
y_pred = eval_result.predictions
residuals = eval_result.residuals

# 1. Predicted vs Actual (scatter)
ax = axes[0, 0]
ax.scatter(y_true, y_pred, alpha=0.4, s=8, c='#2E86AB')
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Price ($)', fontsize=11)
ax.set_ylabel('Predicted Price ($)', fontsize=11)
ax.set_title(f'Predicted vs Actual\nR² = {eval_result.metrics["r2"]:.4f}', 
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Residual Distribution
ax = axes[0, 1]
ax.hist(residuals, bins=60, color='#2E86AB', alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
ax.axvline(residuals.mean(), color='green', linestyle='--', linewidth=2, 
           label=f'Mean: ${residuals.mean():.3f}')
ax.set_xlabel('Residual (Actual - Predicted) ($)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title(f'Residual Distribution\nMAE = ${eval_result.metrics["mae"]:.4f}', 
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Residuals vs Predicted (heteroscedasticity check)
ax = axes[1, 0]
ax.scatter(y_pred, residuals, alpha=0.4, s=8, c='#2E86AB')
ax.axhline(0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Predicted Price ($)', fontsize=11)
ax.set_ylabel('Residual ($)', fontsize=11)
ax.set_title('Residuals vs Predicted\n(Check for heteroscedasticity)', 
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# 4. Q-Q Plot (normality check)
ax = axes[1, 1]
from scipy import stats
stats.probplot(residuals, dist="norm", plot=ax)
ax.set_title('Q-Q Plot\n(Residual Normality Check)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Metrics summary bar chart (key metrics in original or normalized scale)
fig, ax = plt.subplots(figsize=(10, 4))
metrics_to_plot = ['mae', 'rmse', 'r2', 'mape']
labels = ['MAE', 'RMSE', 'R²', 'MAPE (%)']
values = [eval_result.metrics.get(m, 0) for m in metrics_to_plot]
# R² and MAPE need different scaling for visual balance
colors = ['#2E86AB', '#A23B72', '#28a745', '#F18F01']
bars = ax.barh(labels, values, color=colors, alpha=0.8)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel('Value')
ax.set_title('Test Set Metrics Summary')
for bar, val in zip(bars, values):
    ax.text(val + 0.02 if val >= 0 else val - 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 4.3 Error Analysis by Feature

Understanding how errors vary with input features helps identify model weaknesses.

In [ ]:
# Denormalize features for visualization
test_features_orig = test_data.feature_stats.denormalize(test_data.features) if test_data.feature_stats else test_data.features
abs_errors = np.abs(residuals)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Prediction Error Analysis by Feature', fontsize=14, fontweight='bold', y=1.02)

feature_names = ['spot', 'strike', 'volatility', 'rate', 'time_to_expiry', 'is_call']
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B', '#95C623']

for ax, name, color, i in zip(axes.flat, feature_names, colors, range(6)):
    feature_vals = test_features_orig[:, i]
    
    if name == 'is_call':
        # Box plot for categorical
        put_errors = abs_errors[feature_vals < 0.5]
        call_errors = abs_errors[feature_vals >= 0.5]
        ax.boxplot([put_errors, call_errors], labels=['Put', 'Call'])
        ax.set_ylabel('Absolute Error ($)')
    else:
        # Scatter for continuous
        ax.scatter(feature_vals, abs_errors, alpha=0.3, s=5, c=color)
        ax.set_ylabel('Absolute Error ($)')
        
        # Add trend line
        z = np.polyfit(feature_vals, abs_errors, 1)
        p = np.poly1d(z)
        x_sorted = np.sort(feature_vals)
        ax.plot(x_sorted, p(x_sorted), 'r--', linewidth=2, label='Trend')
        ax.legend()
    
    ax.set_xlabel(name.replace('_', ' ').title())
    ax.set_title(f'Error vs {name}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# 5. Inference and Deployment

The inference module provides:

- **SavedModel export**: TensorFlow's production format
- **Model artifacts**: Config, metadata, normalization stats
- **Predictor class**: Efficient batch inference

## 5.1 Saving the Model

In [ ]:
from src.m_learning.inference import save_model, load_model, Predictor
import tempfile
import os

# Create temporary directory for model
model_dir = tempfile.mkdtemp()
model_path = os.path.join(model_dir, 'fx_vanilla_pricer')

# Save model with all artifacts
save_model(
    model=model,
    path=model_path,
    config=config,
    metadata={
        'description': 'FX Vanilla Option Pricer',
        'version': '1.0.0',
        'author': 'QuantStrata',
        'features': dataset.feature_names,
    },
    feature_stats=dataset.feature_stats,
    target_stats=dataset.target_stats,
    training_history=result.history,
)

# List saved artifacts
print("\nSaved Artifacts:")
for root, dirs, files in os.walk(model_path):
    level = root.replace(model_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        print(f"{subindent}{file} ({size:,} bytes)")

## 5.2 Loading the Model

In [ ]:
# Load model artifact
artifact = load_model(model_path)

print("Loaded Model Artifact:")
print(f"  Model: {artifact.model.name}")
print(f"  Config: {artifact.config is not None}")
print(f"  Metadata: {artifact.metadata}")
print(f"  Feature stats: {artifact.feature_stats is not None}")
print(f"  Target stats: {artifact.target_stats is not None}")

## 5.3 Making Predictions

The `Predictor` class handles normalization and denormalization automatically.

In [ ]:
# Create predictor with scalers
predictor = Predictor(
    model=artifact.model,
    feature_scaler=artifact.feature_stats,
    target_scaler=artifact.target_stats,
)

# Define some sample options to price
sample_options = np.array([
    # [spot, strike, vol, rate, expiry, is_call]
    [100, 100, 0.20, 0.05, 0.5, 1],   # ATM call, 6 months
    [100, 100, 0.20, 0.05, 0.5, 0],   # ATM put, 6 months
    [100, 110, 0.25, 0.05, 1.0, 1],   # OTM call, 1 year
    [100, 90, 0.25, 0.05, 1.0, 0],    # OTM put, 1 year
    [100, 100, 0.30, 0.03, 0.25, 1],  # ATM call, 3 months, high vol
    [100, 105, 0.15, 0.05, 2.0, 1],   # Slightly OTM call, 2 years, low vol
], dtype=np.float32)

# Predict (with automatic normalization/denormalization)
predicted_prices = predictor.predict(
    sample_options,
    normalize=True,   # Normalize input features
    denormalize=True, # Denormalize output prices
)

# Display results
print("\n" + "="*70)
print("OPTION PRICING RESULTS")
print("="*70)
print(f"{'Type':<8} {'Spot':>6} {'Strike':>7} {'Vol':>6} {'Rate':>6} {'Expiry':>7} {'Price':>12}")
print("-"*70)

for opt, price in zip(sample_options, predicted_prices):
    opt_type = 'Call' if opt[5] > 0.5 else 'Put'
    print(f"{opt_type:<8} {opt[0]:>6.0f} {opt[1]:>7.0f} {opt[2]:>6.0%} {opt[3]:>6.1%} {opt[4]:>7.2f}y ${price:>10.2f}")

---

# 6. Advanced Topics

## 6.1 Computing Greeks via Automatic Differentiation

TensorFlow's `GradientTape` allows computing option Greeks as derivatives of price with respect to inputs.

In [ ]:
def compute_greeks(model, features, feature_scaler, target_scaler):
    """
    Compute option Greeks: delta, vega, theta, rho (autodiff) and gamma (finite difference of price).
    """
    import numpy as np
    # Normalize features
    features_norm = tf.constant(feature_scaler.normalize(features), dtype=tf.float32)

    # First-order Greeks via GradientTape
    with tf.GradientTape(persistent=True) as tape1:
        tape1.watch(features_norm)
        price_norm = model(features_norm, training=False)
    grads = tape1.gradient(price_norm, features_norm)
    del tape1

    # Gamma via central difference of price: (P(S+h) - 2*P(S) + P(S-h)) / h^2 (no gradient-of-gradient)
    eps = 1e-4
    batch_size = int(features_norm.shape[0])
    bump = tf.reshape(tf.constant([eps] + [0.0] * 5, dtype=tf.float32), (1, 6))
    bump = tf.tile(bump, [batch_size, 1])
    p_center = model(features_norm, training=False).numpy().flatten()
    p_plus = model(features_norm + bump, training=False).numpy().flatten()
    p_minus = model(features_norm - bump, training=False).numpy().flatten()
    gamma_norm = (p_plus - 2.0 * p_center + p_minus) / (eps ** 2)

    # Convert to original scale (avoid div by zero)
    price_std = float(np.asarray(target_scaler.std).flatten()[0]) + 1e-12
    feat_std = np.asarray(feature_scaler.std).flatten() + 1e-12
    grads_np = grads.numpy() if grads is not None else np.zeros((batch_size, 6), dtype=np.float32)

    return {
        'delta': grads_np[:, 0] * price_std / feat_std[0],
        'vega': grads_np[:, 2] * price_std / feat_std[2] / 100,
        'theta': -grads_np[:, 4] * price_std / feat_std[4] / 365,
        'rho': grads_np[:, 3] * price_std / feat_std[3] / 100,
        'gamma': gamma_norm * price_std / (feat_std[0] ** 2),
    }

# Compute Greeks for sample options
greeks = compute_greeks(artifact.model, sample_options, artifact.feature_stats, artifact.target_stats)

print("\n" + "="*80)
print("OPTION GREEKS (via Automatic Differentiation)")
print("="*80)
print(f"{'Type':<6} {'Price':>8} {'Delta':>8} {'Gamma':>8} {'Vega':>8} {'Theta':>8} {'Rho':>8}")
print("-"*80)

for i, (opt, price) in enumerate(zip(sample_options, predicted_prices)):
    opt_type = 'Call' if opt[5] > 0.5 else 'Put'
    print(f"{opt_type:<6} ${price:>7.2f} {greeks['delta'][i]:>8.4f} {greeks['gamma'][i]:>8.4f} "
          f"{greeks['vega'][i]:>8.4f} {greeks['theta'][i]:>8.4f} {greeks['rho'][i]:>8.4f}")

## 6.2 Uncertainty Estimation via MC Dropout

Monte Carlo Dropout provides uncertainty estimates by running multiple forward passes with dropout enabled.

In [ ]:
# Predict with uncertainty (requires dropout in model)
mean_price, std_price = predictor.predict_with_uncertainty(
    sample_options,
    n_samples=100,  # Number of forward passes
    normalize=True,
    denormalize=True,
)

print("\n" + "="*60)
print("PRICES WITH UNCERTAINTY (95% CI)")
print("="*60)
print(f"{'Type':<8} {'Mean Price':>12} {'Std':>10} {'95% CI':>20}")
print("-"*60)

for i, opt in enumerate(sample_options):
    opt_type = 'Call' if opt[5] > 0.5 else 'Put'
    ci_low = mean_price[i] - 1.96 * std_price[i]
    ci_high = mean_price[i] + 1.96 * std_price[i]
    print(f"{opt_type:<8} ${mean_price[i]:>10.2f} ${std_price[i]:>8.3f} [{ci_low:>7.2f}, {ci_high:>7.2f}]")

---

# Summary

This tutorial covered the complete QuantStrata ML pipeline.

**Pipeline at a glance:**  
Data (`TFDataset`) → Model (`tf.keras.Model`) → **Train** (`Trainer.fit`) → **Evaluate** (`evaluate_model`) → **Save/Load** (`save_model`, `load_model`) → **Predict** (`Predictor.predict`). The same flow supports any plug-and-play model (see the "Plug-and-Play Design" section).

| Component | Class/Function | Purpose |
|-----------|----------------|--------|
| **Data** | `TFDataset`, `create_pricing_dataset` | Data loading, normalization, splitting |
| **Model** | `MLPPricer`, `create_mlp_pricer` | Neural network architecture |
| **Config** | `TrainingConfig`, `OptimizerConfig` | Training configuration |
| **Training** | `Trainer`, `fit_model` | Model training with callbacks |
| **Evaluation** | `Evaluator`, `evaluate_model` | Metrics and visualization |
| **Inference** | `save_model`, `load_model`, `Predictor` | Deployment and prediction |

## Key Takeaways

1. **TensorFlow-native**: All components use `tf.data.Dataset`, `tf.keras.Model`, and TensorFlow's SavedModel format

2. **Plug-and-play**: Any `tf.keras.Model` (or `BaseModel` subclass) works with `Trainer`, `Evaluator`, `save_model`/`load_model`, and `Predictor` — same pipeline, different architectures.

3. **Normalization matters**: Always normalize features and targets for stable training

4. **Early stopping**: Prevents overfitting by monitoring validation loss

5. **SavedModel format**: Production-ready serialization with all artifacts

6. **Greeks via autodiff**: TensorFlow's GradientTape enables automatic Greek computation

## Next Steps

- Try different model architectures (`ResidualMLPPricer`)
- Experiment with hyperparameters (learning rate, hidden units)
- Train on real market data
- Explore the GNN-RNN hybrid model for portfolio pricing

See `docs/architecture/component_reference.md` for the full API reference.

In [ ]:
# Cleanup
import shutil
shutil.rmtree(model_dir, ignore_errors=True)

print("\nTutorial complete!")